# Pose-Controlled Image Generation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tigermorning/pose-image-tool/blob/main/pose_tool.ipynb)

Generate images that follow **both** a reference pose and a text prompt.

**Pipeline:** reference photo -> OpenPose keypoint detection -> skeleton image -> ControlNet -> SDXL -> output image.

| | |
|---|---|
| Base model | `stabilityai/stable-diffusion-xl-base-1.0` |
| ControlNet | `thibaud/controlnet-openpose-sdxl-1.0` |
| Pose annotator | `controlnet_aux.OpenposeDetector` (`lllyasviel/Annotators`) |
| GPU | Colab free **T4** is enough (fp16, ~10 GB peak). Placement adapts to the VRAM and RAM the notebook measures. |
| Runtime | ~5 min setup + ~50 s per 832x1216 image |

**Before you start:** `Runtime > Change runtime type > T4 GPU`, and have 2 photos of people in clearly different poses ready to upload (section 3).

## How this notebook answers the assignment

| Assignment | Where | Output |
|---|---|---|
| **Step 1** - one reference photo, extract the pose, generate a *different person* in that same pose | section 5, prompt **A1** | `samples/output_01.png` |
| **Step 2a** - same pose, change only the prompt | section 5, **A1 vs A2** | `output_01.png`, `output_01_alt_prompt.png` |
| **Step 2b** - same prompt, change only the pose photo | section 5, **B1 vs B2** | `output_01.png`, `output_02.png` |
| **Step 2** write-up - what was changed, how the output changed | section 7 | - |

`output_01.png` and `output_02.png` share prompt and seed exactly and differ only in which photograph
the skeleton came from. That difference is the evidence the tool works.

**Sections:** 1 Install dependencies - 2 Load models - 3 Extract pose - 4 Generate image - 5 Experiments - 6 Save results - 7 Findings

---
## 1. Install dependencies

In [ ]:
# Install the diffusion stack and the OpenPose annotator.
# controlnet_aux is pinned to 0.0.10: the older 0.0.9 constrains timm so tightly that pip
# cannot resolve it against a current timm, while 0.0.10 imports cleanly on timm 1.x.
#
# mediapipe is deliberately NOT installed here. controlnet_aux imports its mediapipe_face
# module at package level, and against the mediapipe build currently on Colab that raises
#   AttributeError: module 'mediapipe' has no attribute 'solutions'
# which breaks `from controlnet_aux import OpenposeDetector` outright. Without mediapipe
# installed, controlnet_aux only prints a warning and the OpenPose path works normally.
!pip install -q "diffusers>=0.31" "transformers>=4.44" "accelerate>=0.34" safetensors
!pip install -q "controlnet_aux==0.0.10"

In [ ]:
# Confirm a CUDA GPU is attached, then measure BOTH memory ceilings before anything loads.
# Colab enforces them separately and they fail differently: exhausting system RAM kills the
# session outright, exhausting VRAM raises CUDA out of memory. The placement decision in the
# next cell reads these numbers, so they are measured rather than assumed.
import platform
import shutil

import torch

assert torch.cuda.is_available(), 'No GPU attached. Colab: Runtime > Change runtime type > T4 GPU.'
_props = torch.cuda.get_device_properties(0)
FREE_VRAM_GB, TOTAL_VRAM_GB = (b / 1024 ** 3 for b in torch.cuda.mem_get_info())

try:
    import psutil

    FREE_RAM_GB = psutil.virtual_memory().available / 1024 ** 3
    TOTAL_RAM_GB = psutil.virtual_memory().total / 1024 ** 3
except ImportError:              # psutil ships with Colab, but do not hard-fail without it
    FREE_RAM_GB = TOTAL_RAM_GB = float('nan')

print(f'python : {platform.python_version()}')
print(f'torch  : {torch.__version__}')
print(f'gpu    : {_props.name}')
print(f'vram   : {FREE_VRAM_GB:.1f} GB free of {TOTAL_VRAM_GB:.1f} GB')
print(f'ram    : {FREE_RAM_GB:.1f} GB free of {TOTAL_RAM_GB:.1f} GB')
print(f'disk   : {shutil.disk_usage(".").free / 1024 ** 3:.1f} GB free')

# T4 has no usable bf16 path, so fp16 is used for every module in this notebook.
DTYPE = torch.float16

---
## 2. Load models

Two independent model groups are loaded here:

1. **OpenPose annotator** - detects human keypoints. It only *reads* images, it generates nothing.
2. **SDXL + ControlNet** - generates images. ControlNet is the adapter that injects the skeleton into every denoising step.

In [ ]:
# MODEL LOAD 1 of 2 - the pose detector.
# Load the OpenPose annotator (body + hand + face keypoint models from lllyasviel/Annotators).
import numpy as np
from controlnet_aux import OpenposeDetector
# drawing helpers, used to render the hint at the exact generation resolution
from controlnet_aux.open_pose.util import draw_bodypose, draw_handpose

# Guarded so re-running this cell does not load a second copy alongside the first.
if 'openpose' not in globals():
    openpose = OpenposeDetector.from_pretrained('lllyasviel/Annotators')
print('OpenPose annotator ready')

In [ ]:
# MODEL LOAD 2 of 2 - the image generator.
# Loads SDXL base plus the OpenPose ControlNet. The fp16-fix VAE is mandatory: the stock SDXL
# VAE overflows in fp16 and returns black images.
#
# Guarded by `if 'pipe' in globals()`. SDXL plus ControlNet is roughly 10 GB, and re-running this
# cell unguarded keeps the first copy alive while a second loads - the usual way a free Colab
# session ends with "used all available RAM" partway through the experiments.
import gc

from diffusers import AutoencoderKL, ControlNetModel, StableDiffusionXLControlNetPipeline

BASE_ID = 'stabilityai/stable-diffusion-xl-base-1.0'
CONTROLNET_ID = 'thibaud/controlnet-openpose-sdxl-1.0'

# Rough fp16 footprint of this pipeline at 832x1216: UNet ~5.1 GB, ControlNet ~2.5 GB, the two
# text encoders ~1.5 GB, VAE ~0.2 GB, plus activations. 11 GB is that total with headroom.
VRAM_NEEDED_GB = 11.0
RAM_NEEDED_FOR_OFFLOAD_GB = 8.0

if 'pipe' in globals():
    print('pipeline already loaded - skipping')
else:
    controlnet = ControlNetModel.from_pretrained(CONTROLNET_ID, torch_dtype=DTYPE)
    vae = AutoencoderKL.from_pretrained('madebyollin/sdxl-vae-fp16-fix', torch_dtype=DTYPE)
    pipe = StableDiffusionXLControlNetPipeline.from_pretrained(
        BASE_ID,
        controlnet=controlnet,
        vae=vae,
        torch_dtype=DTYPE,
        variant='fp16',
        use_safetensors=True,
    )

    # Placement is chosen from both ceilings, not just VRAM. The trap worth knowing: offloading
    # trades VRAM for system RAM, so on a RAM-starved VM it makes the more dangerous failure more
    # likely rather than less. When the GPU can hold everything - a 15 GB T4 can - keeping it
    # resident is both the fastest option and the one that leaves system RAM alone.
    if FREE_VRAM_GB >= VRAM_NEEDED_GB:
        pipe.to('cuda')
        placement = f'full GPU ({FREE_VRAM_GB:.1f} GB VRAM free, nothing held in system RAM)'
    elif FREE_RAM_GB >= RAM_NEEDED_FOR_OFFLOAD_GB:
        # Whole modules move to the GPU as each is called, then back. Slower than resident, but
        # far faster than sequential offload, which the diffusers docs call extremely slow.
        pipe.enable_model_cpu_offload()
        pipe.enable_vae_tiling()      # decodes the latent in overlapping tiles: lowers peak VRAM
        placement = f'model CPU offload ({FREE_VRAM_GB:.1f} GB VRAM, {FREE_RAM_GB:.1f} GB RAM free)'
    else:
        # Both ceilings are tight. Sequential offload moves submodules rather than whole models:
        # the smallest VRAM footprint available, and by far the slowest.
        pipe.enable_sequential_cpu_offload()
        pipe.enable_vae_tiling()
        placement = 'sequential CPU offload - LAST RESORT, expect minutes per image'

    # No enable_vae_slicing() here, on purpose. Slicing splits a *batch* of latents, and the
    # diffusers docs state there is no performance impact for single-image batches. This notebook
    # generates one image at a time, so tiling is the setting that actually lowers peak memory.
    pipe.set_progress_bar_config(leave=False)

    # Drop the CPU-side copies left behind by loading.
    gc.collect()
    torch.cuda.empty_cache()
    print(f'pipeline ready - {placement}')

---
## 3. Extract pose

Upload the reference photos, then convert each one into a skeleton image.

- **Experiment A** (same pose, different prompts) needs **1** reference.
- **Experiment B** (same prompt, different poses) needs **2+** references with visibly different poses.

Good references: one person, full body or at least head-to-knee, limbs not overlapping the torso. Crowds and heavy occlusion are where OpenPose fails.

In [ ]:
# Upload the reference photos (jpg / png / webp). They are kept in ./references,
# and every artifact this notebook produces goes to ./samples.
from pathlib import Path

from PIL import Image, ImageOps

REF_DIR = Path('references')
OUT_DIR = Path('samples')
REF_DIR.mkdir(exist_ok=True)
OUT_DIR.mkdir(exist_ok=True)

try:
    from google.colab import files

    for fname, data in files.upload().items():
        (REF_DIR / fname).write_bytes(data)
except ImportError:
    print('Not running on Colab - copy your images into ./references by hand, then re-run this cell.')

SUFFIXES = {'.jpg', '.jpeg', '.png', '.webp'}
REFS = sorted(p for p in REF_DIR.iterdir() if p.suffix.lower() in SUFFIXES)
assert REFS, 'No reference images found in ./references'
print(f'{len(REFS)} reference image(s):', [p.name for p in REFS])

In [ ]:
# Helpers. The ControlNet hint and the generated image must have identical dimensions,
# so every reference is cropped to one fixed portrait resolution up front.
WIDTH, HEIGHT = 832, 1216  # SDXL-native portrait ratio; lighter on a T4 than 1024x1024


def load_ref(path, width=WIDTH, height=HEIGHT):
    """Open a reference photo, apply its EXIF rotation, and letterbox it to the target ratio.

    Letterboxing, not centre-cropping. A crop silently removes whatever falls outside the target
    aspect ratio: on one of the references here it cut the left ankle out of frame, which cost a
    keypoint and disabled pose control for that leg. Padding keeps the whole body.
    """
    img = ImageOps.exif_transpose(Image.open(path)).convert('RGB')
    return ImageOps.pad(img, (width, height), method=Image.LANCZOS)

In [ ]:
# THE POSE EXTRACTION STEP. Detects the person's joints and draws them as the skeleton
# image that ControlNet reads. This is the only cell that looks at the reference photo's
# content; everything after it works from the skeleton alone.
def extract_pose(image, include_hand=True, detect_resolution=None):
    """Run OpenPose on a photo and draw the skeleton hint at exactly the target resolution.

    The detector's own __call__ renders at its internal resolution and rescales, which both
    blurs the stick figure and changes its aspect ratio (832x1216 in, 832x1280 out). A blurred,
    vertically squashed hint is an ambiguous hint. So the keypoints are taken directly and drawn
    once, crisply, on a canvas that already matches the image being generated.
    """
    width, height = image.size
    results = openpose.detect_poses(np.array(image),
                                    include_hand=include_hand, include_face=False)
    if not results:
        raise RuntimeError('OpenPose found no person in this image.')

    canvas = np.zeros((height, width, 3), dtype=np.uint8)
    pose = results[0]
    canvas = draw_bodypose(canvas, pose.body.keypoints)
    if include_hand:
        canvas = draw_handpose(canvas, pose.left_hand)
        canvas = draw_handpose(canvas, pose.right_hand)
    return Image.fromarray(canvas)

In [ ]:
# RUN THE EXTRACTION over every uploaded reference.
# Extract one skeleton per reference and save it as the repository's pose_NN.png artifact.
poses = []
for i, path in enumerate(REFS, start=1):
    ref = load_ref(path)
    skeleton = extract_pose(ref)
    skeleton.save(OUT_DIR / f'pose_{i:02d}.png')
    poses.append({'id': f'{i:02d}', 'source': path.name, 'ref': ref, 'pose': skeleton})

print('saved:', [f"pose_{p['id']}.png" for p in poses])

### 3a. Check the hints before generating

**Diagnostic, but do not skip it.** A skeleton can look plausible and still be unusable: if a hip-to-ankle chain is not closed, pose control over that limb is silently off and nothing later in the run will tell you. This cell turns that into a number.

In [ ]:
# Validate each skeleton NUMERICALLY before spending GPU time on it.
# A skeleton image can look plausible while being topologically wrong: the hip-knee-ankle
# chains may not close into two legs. That is invisible by eye and produces extra or fused
# limbs in the output, so it is checked explicitly here.
COCO18 = ['nose', 'neck', 'r_shoulder', 'r_elbow', 'r_wrist', 'l_shoulder', 'l_elbow', 'l_wrist',
          'r_hip', 'r_knee', 'r_ankle', 'l_hip', 'l_knee', 'l_ankle', 'r_eye', 'l_eye',
          'r_ear', 'l_ear']
LEG_CHAINS = {'right leg': (8, 9, 10), 'left leg': (11, 12, 13)}
ARM_CHAINS = {'right arm': (2, 3, 4), 'left arm': (5, 6, 7)}


def validate_pose(image, label=''):
    """Report detected keypoints and whether every limb chain is complete.

    Returns True only if one person was found and all four limb chains are closed.
    """
    try:
        # detect_poses works on a numpy array, not a PIL image
        results = openpose.detect_poses(np.array(image))
    except Exception as exc:  # detect_poses is not in every controlnet_aux release
        print(f'{label}: numeric validation unavailable ({type(exc).__name__}) - inspect by eye')
        return None

    if len(results) != 1:
        print(f'{label}: FAIL - {len(results)} people detected, this pipeline assumes exactly 1')
        return False

    keypoints = results[0].body.keypoints
    found = {i for i, kp in enumerate(keypoints) if kp is not None}
    missing = [COCO18[i] for i in range(18) if i not in found]

    broken = [name for name, chain in {**LEG_CHAINS, **ARM_CHAINS}.items()
              if not set(chain) <= found]
    print(f'{label}: {len(found)}/18 keypoints')
    if missing:
        print(f'  missing: {", ".join(missing)}')
    if broken:
        print(f'  INCOMPLETE CHAINS: {", ".join(broken)}')
        print('  -> the hint cannot constrain those limbs; expect invented or duplicated ones')
    else:
        print('  all four limb chains complete')

    return not broken


print('Validating hints. Replace any reference that fails before generating.')
print()
for entry in poses:
    entry['valid'] = validate_pose(entry['ref'], f"pose_{entry['id']} ({entry['source']})")
    print()

In [ ]:
# Inspect every reference next to its skeleton. Do this before generating anything:
# a missing arm or a merged pair of legs here will be a missing arm in the output too.
import matplotlib.pyplot as plt
import numpy as np


def show_grid(images, titles, cols=None, size=3.0):
    """Display images in a labelled grid (used for every preview and experiment below)."""
    cols = cols or len(images)
    rows = -(-len(images) // cols)
    fig, axes = plt.subplots(rows, cols, figsize=(size * cols, size * 1.45 * rows), squeeze=False)
    flat = axes.ravel()
    for ax, img, title in zip(flat, images, titles):
        ax.imshow(img)
        ax.set_title(title, fontsize=9)
    for ax in flat:
        ax.axis('off')
    plt.tight_layout()
    plt.show()
    plt.close(fig)   # otherwise every call leaves a figure open


show_grid(
    [im for p in poses for im in (p['ref'], p['pose'])],
    [t for p in poses for t in (p['source'], f"pose_{p['id']}.png")],
    cols=min(4, 2 * len(poses)),
)

### 3c. Optional fallback extractor

Skip this cell if section 3 worked.

`controlnet_aux` is lightly maintained and its imports break whenever `timm` or `huggingface_hub` shift. If the annotator refuses to load, this cell rebuilds an equivalent hint from **MediaPipe Pose** landmarks, remapped to the COCO-18 keypoint order and drawn with the standard OpenPose limb colours - which is what the ControlNet was actually trained to read. Then use `extract_pose_mediapipe` in place of `extract_pose` above.

Trade-off: MediaPipe gives body keypoints only (no hands, no face) and is less accurate on strongly foreshortened limbs.

In [ ]:
# Fallback pose extractor: MediaPipe landmarks -> OpenPose-style COCO-18 skeleton.
# Needs `!pip install mediapipe` first - but note that installing mediapipe can break
# `from controlnet_aux import ...` in the same session, so use this in a fresh runtime
# where the primary path has already failed for another reason.
import cv2
import numpy as np

# MediaPipe landmark index -> COCO-18 index. Neck (COCO 1) has no MediaPipe equivalent
# and is derived from the shoulder midpoint below.
_MP_TO_COCO18 = {0: 0, 2: 15, 5: 14, 7: 17, 8: 16, 11: 5, 12: 2, 13: 6, 14: 3,
                 15: 7, 16: 4, 23: 11, 24: 8, 25: 12, 26: 9, 27: 13, 28: 10}
_LIMBS = [(1, 2), (1, 5), (2, 3), (3, 4), (5, 6), (6, 7), (1, 8), (8, 9), (9, 10),
          (1, 11), (11, 12), (12, 13), (1, 0), (0, 14), (14, 16), (0, 15), (15, 17)]
_COLORS = [(255, 0, 0), (255, 85, 0), (255, 170, 0), (255, 255, 0), (170, 255, 0), (85, 255, 0),
           (0, 255, 0), (0, 255, 85), (0, 255, 170), (0, 255, 255), (0, 170, 255), (0, 85, 255),
           (0, 0, 255), (85, 0, 255), (170, 0, 255), (255, 0, 255), (255, 0, 170), (255, 0, 85)]


def extract_pose_mediapipe(image, min_visibility=0.4):
    """Drop-in replacement for extract_pose() that does not depend on controlnet_aux."""
    import mediapipe as mp

    width, height = image.size
    with mp.solutions.pose.Pose(static_image_mode=True, model_complexity=2) as detector:
        result = detector.process(np.array(image))
    if not result.pose_landmarks:
        raise RuntimeError('MediaPipe found no person in this image.')

    points = [None] * 18
    for mp_index, coco_index in _MP_TO_COCO18.items():
        landmark = result.pose_landmarks.landmark[mp_index]
        if landmark.visibility >= min_visibility:
            points[coco_index] = (int(landmark.x * width), int(landmark.y * height))
    if points[2] and points[5]:  # neck = midpoint between the two shoulders
        points[1] = ((points[2][0] + points[5][0]) // 2, (points[2][1] + points[5][1]) // 2)

    canvas = np.zeros((height, width, 3), dtype=np.uint8)
    for i, (a, b) in enumerate(_LIMBS):
        if points[a] and points[b]:
            cv2.line(canvas, points[a], points[b], _COLORS[i], 4)
    for i, point in enumerate(points):
        if point:
            cv2.circle(canvas, point, 4, _COLORS[i], -1)
    return Image.fromarray(canvas)

---
## 4. Generate image

One helper does all generation. Every experiment below only changes its arguments, so the variable under test is always explicit.

Key knobs:

| Argument | Effect |
|---|---|
| `conditioning_scale` | How hard the skeleton is enforced. Low = prompt wins, high = pose wins but anatomy stiffens. |
| `guidance_scale` | How hard the text prompt is enforced. |
| `seed` | Fixed by default so experiments stay single-variable. |

`conditioning_scale` defaults to **1.0**, which is what measurement supports: at 0.8 a folded knee landed 0.45 of the frame from the hint, at 1.0 every joint was within 0.07.

In [ ]:
# THE IMAGE GENERATION STEP.
# Single entry point for generation: conditions on the pose skeleton and the text prompt together.
# 'three legs' and 'duplicated limbs' are here because an incoherent pose hint made SDXL
# render several interpretations of the same limb at once - see section 7.
NEGATIVE = ('lowres, blurry, deformed hands, extra fingers, extra limbs, three legs, '
            'duplicated limbs, fused limbs, watermark, text, jpeg artifacts')


def generate(pose_image, prompt, seed=1234, steps=28, guidance_scale=5.0,
             conditioning_scale=1.0, negative_prompt=NEGATIVE):
    """Return one image that follows both `pose_image` (via ControlNet) and `prompt` (via SDXL).

    The seed is an explicit argument so any two calls can be compared as a controlled experiment.

    conditioning_scale defaults to 1.0 because that is what measurement supports: at 0.8 the
    folded knee in this notebook's reference landed 0.45 of the frame away from the hint, at 1.0
    every joint is within 0.07. The cost is slightly stiffer material rendering.
    """
    torch.cuda.empty_cache()   # keep VRAM from fragmenting across a run
    generator = torch.Generator('cuda').manual_seed(seed)
    return pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        image=pose_image,
        width=pose_image.width,
        height=pose_image.height,
        num_inference_steps=steps,
        guidance_scale=guidance_scale,
        controlnet_conditioning_scale=conditioning_scale,
        generator=generator,
    ).images[0]

### 4a. Score pose fidelity

**Diagnostic.** Re-detects the pose in the generated image and compares it with the hint, joint by joint. Roughly 0.03 means the pose was reproduced; 0.3 means a different pose. Judging this by eye is unreliable - a folded leg can look fine while sitting a third of the frame away from the hint.

In [ ]:
# Measure pose fidelity objectively: re-detect the pose in the GENERATED image and compare it
# joint by joint with the hint it was conditioned on. Eyeballing a skeleton against a photo is
# unreliable - a folded leg can look plausible while sitting 0.4 of the frame away from the hint.
def pose_fidelity(hint_keypoints, image, label=''):
    """Report mean and max joint displacement between the hint and the generated image.

    Distances are normalised to the image (0.03 is a close match, 0.3 is a different pose).
    Only the limbs are scored; head keypoints follow the prompt more than the hint.
    """
    detected = openpose.detect_poses(np.array(image))
    if not detected:
        print(f'{label}: no person detected in the output')
        return None

    got = detected[0].body.keypoints
    scored, worst = [], ('', 0.0)
    for i in (2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13):
        a, b = hint_keypoints[i], got[i]
        if a is None or b is None:
            continue
        distance = ((a.x - b.x) ** 2 + (a.y - b.y) ** 2) ** 0.5
        scored.append(distance)
        if distance > worst[1]:
            worst = (COCO18[i], distance)
    if not scored:
        print(f'{label}: no comparable joints')
        return None
    mean = sum(scored) / len(scored)
    print(f'{label}: mean joint error {mean:.3f}, worst {worst[0]} {worst[1]:.3f} '
          f'({len(scored)} joints)')
    return mean


# Keypoints of each hint, kept so generated images can be scored against them.
for entry in poses:
    detected = openpose.detect_poses(np.array(entry['ref']))
    entry['keypoints'] = detected[0].body.keypoints if detected else None
print('hint keypoints stored for fidelity scoring')

---
## 5. Experiments - the assignment's step 1 and step 2

Two runs, each isolating one variable. Prompt A1 doubles as step 1's deliverable: it is the
*different person in the reference's pose*. Prompts, seeds and settings are recorded in
`prompts.md`.

### Step 1 and Step 2a - same pose, different prompts

**A1 is step 1**: the reference is a seated man, and this prompt asks for a young African woman. Whatever posture survives comes from the skeleton alone, because the prompt never mentions one.

**A1 against A2 is step 2a**: the hint, the seed and every sampler setting are held fixed, so any difference between the two images is attributable to the prompt text and nothing else.

In [ ]:
# EXPERIMENT A - one pose, two prompts, identical seed and sampler settings.
# A1 is the assignment's step 1: a different person in the reference's pose. A2 changes the
# medium as well as the subject, which is the harder test - moving away from photorealism is
# where pose adherence weakens first.
# Neither prompt names a posture. The skeleton supplies that, and A1 is reused in experiment B
# over a standing hint, where a posture word would fight the hint and double the limbs.
POSE_A = poses[0]['pose']
SEED = 1234                     # one seed for both experiments, so every comparison is single-variable
PROMPTS_A = [
    'A beautiful young African woman, short natural hair, small gold hoop earrings, wearing a short bohemian dress in deep teal with gold embroidery, bare legs and flat leather sandals, in a sunlit concrete gallery with tall arched windows, warm late-afternoon light raking from the left, 85mm lens at f/2, editorial fashion photograph, fine skin texture',
    'An ink and watercolour drawing of a travelling puppeteer in a patched wool coat and worn leather boots, a canvas satchel at the hip, on cream paper with visible fibres, loose confident brush strokes, muted earth pigments of ochre and burnt umber, ink bleeding at the outlines, wide untouched margins',
]

exp_a = []
for i, prompt in enumerate(PROMPTS_A, start=1):
    image = generate(POSE_A, prompt, seed=SEED)
    exp_a.append(image)
    pose_fidelity(poses[0]['keypoints'], image, f'A{i}')

show_grid([POSE_A, *exp_a], [f"pose_{poses[0]['id']}.png (fixed)", 'A1', 'A2'])

### Step 2b - same prompt, different pose photo

The prompt is A1 verbatim and the seed is fixed. The only thing that changes is which photograph the skeleton was extracted from: `pose_01` (seated on a stool) or `pose_02` (standing, fists raised). This is the inverse of step 2a.

In [ ]:
# EXPERIMENT B - one prompt, two poses, identical seed and sampler settings.
# The prompt is PROMPTS_A[0] and the seed is the same SEED, so the pose_01 arm of this
# comparison is byte-identical to A1 and is not regenerated - exp_a[0] is reused directly.
# Only the remaining hints need generating, which keeps this a true single-variable test:
# prompt, seed and every sampler setting are held, and only the skeleton changes.
assert len(poses) >= 2, 'Upload at least two references with different poses to run experiment B.'
PROMPT_B = PROMPTS_A[0]

exp_b = [exp_a[0]]                                   # pose_01 arm, already generated as A1
for entry in poses[1:]:
    image = generate(entry['pose'], PROMPT_B, seed=SEED)
    pose_fidelity(entry['keypoints'], image, f"B on pose_{entry['id']}")
    exp_b.append(image)

used = poses[:len(exp_b)]
show_grid([e['pose'] for e in used] + exp_b,
          [f"pose_{e['id']}" for e in used] + [f"B on pose_{e['id']}" for e in used],
          cols=len(used))

---
## 6. Save results

Writes the four filenames the repository expects, then zips `samples/` for download so the images can be committed.

In [ ]:
# THE RESULT SAVING STEP.
# Writes only files that carry information nothing else does, under the names the repository
# expects. output_01 is A1 (step 1) and output_02 is the same prompt over the second pose
# (step 2b); output_01_alt_prompt is A2, the other arm of step 2a. The skeletons were already
# saved in section 3.
import shutil

exp_a[0].save(OUT_DIR / 'output_01.png')             # step 1: a different person in the reference pose
exp_a[1].save(OUT_DIR / 'output_01_alt_prompt.png')  # step 2a: same pose, the other prompt
exp_b[1].save(OUT_DIR / 'output_02.png')             # step 2b: same prompt and seed, the other pose

shutil.make_archive('samples', 'zip', root_dir='.', base_dir='samples')  # cross-platform, no shell
print('samples/:', sorted(p.name for p in OUT_DIR.iterdir()))

try:
    from google.colab import files as colab_files

    colab_files.download('samples.zip')
except ImportError:
    print('samples.zip written next to the notebook')

# What the run actually cost, so the next person knows whether they had headroom.
print(f'peak VRAM: {torch.cuda.max_memory_allocated() / 1024 ** 3:.1f} GB')
try:
    import psutil

    print(f'RAM still free: {psutil.virtual_memory().available / 1024 ** 3:.1f} GB')
except ImportError:
    pass

---
## 7. Findings

Everything below was measured with the pipeline exactly as this notebook now runs it: the hint drawn
at the generation resolution, references letterboxed rather than cropped, hand keypoints enabled,
`controlnet_conditioning_scale` 1.0, `guidance_scale` 5.0, 28 steps, seed 1234, 832x1216.

Earlier drafts of this notebook had three defects that produced badly wrong images - a blurred and
vertically squashed hint, a centre crop that removed keypoints, and a conditioning scale too low to
hold a folded limb. Those outputs are not part of this repository and nothing here describes them as
results. They appear below only where the fix is worth documenting.

### What was changed

| Run | Held fixed | Varied |
|---|---|---|
| **Step 1 / 2a** | pose hint `pose_01`, seed 1234, 28 steps, guidance 5.0, conditioning 1.0 | the prompt: a photographic African-woman brief, then an ink-and-watercolour brief |
| **Step 2b** | the prompt (step 1's, verbatim), seed 1234, every sampler setting | the pose hint: `pose_01` seated, `pose_02` standing |

Both hints passed section 3a at 18/18 keypoints with all four limb chains closed and no limbs
crossing in 2D. That is the first reference pair in this project to have no defect at all; four
earlier candidates were rejected by the same check.

### What changed in the output

Each figure is the `pose_fidelity()` score printed by section 5: the pose re-detected in the
generated image and compared with the hint joint by joint, over the twelve limb joints. Roughly 0.03
means the pose was reproduced, 0.3 means a different pose.

| Output | Mean joint error | Worst joint | What it shows |
|---|---|---|---|
| `output_01.png` | **0.022** | r_ankle 0.042 | step 1 - a different person in the reference's pose |
| `output_01_alt_prompt.png` | 0.046 | r_ankle 0.160 | step 2a - same hint and seed, different prompt |
| `output_02.png` | 0.045 | r_knee 0.144 | step 2b - same prompt and seed, different hint |

**Step 1 worked.** `output_01.png` reproduces the reference pose to within 0.042 on every one of the
twelve joints - one leg extended to the ground, the other knee raised, both feet visible in sandals,
one hand at the hip and the other resting on the leg, the head turned as in the photograph. The
subject is a different person entirely. That is the claim the assignment asks for, and it is legible
without needing the score.

**Step 2a - changing only the prompt.** The hint, the seed and every sampler setting were held. Both
outputs keep the posture; everything the skeleton does not encode moved completely - person,
wardrobe, setting, palette, and medium, since the second prompt left photography for ink and
watercolour. The pose survives the medium change at 0.046, worse than 0.022 but still clearly the
same posture. **Moving away from photorealism costs pose fidelity without breaking it.**

**Step 2b - changing only the pose photo.** Identity, wardrobe and palette carried across both
images while posture followed whichever photograph the skeleton came from: seated in `output_01.png`,
standing with the forearms raised in `output_02.png`. Since prompt and seed are byte-identical
between those two files, the difference between them is attributable to the hint and nothing else.

**Which change matters more.** Changing the prompt rewrote the entire image except the posture.
Changing the hint rewrote only the posture. That asymmetry is the finding: **the skeleton owns where
the body is, the prompt owns what the body is.**

One honest wrinkle. The same prompt asked for a short dress with bare legs, and the seated pose got
one while the standing pose was given a longer dress that covered the calves and hid the feet. The
worst joint in `output_02.png` is the knee, at 0.144, and that is where the fabric is. Garment length
is not reliably controllable from the prompt, and where clothing covers a joint the score degrades
because the re-detected keypoint has to be inferred.

### What the settings are, and why

Each of these came from a controlled comparison with everything else held constant.

| Setting | Evidence |
|---|---|
| **Draw the hint at the generation resolution** | `OpenposeDetector.__call__` renders at its own internal resolution and rescales the result: 832x1216 in, 832x**1280** out. Rescaling back blurred the stick figure from ~43 distinct colours to 2210 and squashed the body 5% vertically, and ControlNet, unable to tell which shin belonged to which thigh, drew **three legs**. Taking keypoints from `detect_poses()` and drawing them once at the target size removed the extra limb with prompt, seed and scale unchanged. Hint sharpness is part of the conditioning signal, not cosmetics. |
| **Letterbox, do not centre-crop** | `ImageOps.fit` discarded 12% of the height of a 355x592 reference and took its left ankle out of frame: 18/18 keypoints down to 17/18. `ImageOps.pad` keeps the whole body. |
| **`conditioning_scale` 1.0, not 0.8** | At 0.8 a folded knee landed 0.448 of the frame from the hint (mean 0.099); at 1.0 every joint was within 0.066 (mean 0.031). The torso and arms matched at both scales, so the failure was invisible without measuring. |
| **Keep hand keypoints enabled** | Common guidance for this ControlNet says to disable hand detection. Measured with everything else held: the body-only hint scored mean 0.108 against 0.085 and slid the braced wrist 0.278 out of place. Hand keypoints anchor the wrist even though they do not fix the fingers. |
| **Choose placement from RAM as well as VRAM** | Offloading trades VRAM for system RAM, so on a RAM-limited VM it makes the failure that kills the session more likely, not less. A 15 GB T4 holds the whole pipeline, making the fast path also the safe one. |
| **`enable_vae_tiling()`, not `enable_vae_slicing()`** | Slicing splits a batch of latents; the diffusers docs state no performance impact for single-image batches, which is all this notebook generates. Tiling is what lowers peak memory for one large image. |

### Limitations observed

1. **A perfect skeleton is not sufficient.** A hint can score 18/18 and still fail if it is rendered
   at the wrong size or blurred. Detection quality and hint quality are separate problems.
2. **Detection failure is silent.** A broken hint does not produce a broken pose, it produces a
   *default* pose: with nothing coherent to enforce, ControlNet contributes nothing and the model
   falls back on its own prior. Nothing in the run reports it, which is why section 3a exists.
3. **Occlusion is what breaks detection.** An arm hidden inside a coat cost a whole limb chain
   (15/18); an upper-body crop had no leg keypoints at all (12/18).
4. **The hint is 2D.** No depth is encoded, so overlapping limbs cannot be disambiguated. Facing
   direction has to come from the prompt.
5. **Fingers on a raised hand come out merged.** Three hint variants were compared with everything
   else held constant - body plus hand keypoints, body only, and body only with thicker limb lines -
   and all three failed. Hands occupy too few pixels at 832x1216. An inpainting pass over the hand
   region is the real fix and is outside this tool's scope. Hands resting on a surface come out
   clearly better than raised ones, which is why `pose_02` uses closed fists.
6. **Clothing can hide the very thing being demonstrated.** A floor-length dress covered the legs
   completely, which made the pose unverifiable in the output and degraded the fidelity score because
   the ankles had to be guessed. Same hint and seed, the ink-drawing prompt scored 0.046 and the
   floor-length one 0.098. The garment now leaves the legs and feet visible.
7. **Framing is not independent of the pose.** The crop follows the skeleton's extent in frame.
8. **Prompt clauses are not honoured equally.** Concrete beats abstract - a clause naming a material
   and an optical effect is rendered, a bare abstract noun tends to be ignored. Each text encoder
   also truncates at 77 tokens silently, from the tail, so any prompt edit needs re-checking against
   the tokenizer.
9. **Single subject only.** Multi-person skeletons are detected but SDXL blends identities between
   overlapping figures. Not exercised here.
10. **Realism is SDXL 1.0's ceiling.** A photoreal fine-tune is a drop-in `BASE_ID` swap and would
    help; matching a frontier image model needs a different base model, not different settings.
11. **Not FLUX.** SDXL was chosen so this runs on a free T4. FLUX.2-klein is 9B with a 24B text
    encoder and has no official ControlNet; FLUX.1-dev has a usable pose ControlNet but needs a paid
    runtime plus quantisation.

### Reproducibility

Fixed seeds reproduce a comparison on the same GPU and library versions. Exact pixels are not
portable across GPUs or `diffusers` versions, since cuDNN kernel selection and fp16 accumulation
order differ. Every prompt, seed and setting is in `prompts.md`.